# Quickstart: the theorem in action on a worked example

<a href="https://colab.research.google.com/github/kootru-repo/cumulant-residual-cert/blob/main/notebooks/01_quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200"/></a>


A concrete numerical walkthrough with **expected vs actual** at every step. You will:

1. Build a small correlated $U(1)$-invariant state on $n = 4$ spin-orbitals.
2. Compute the **true** expectation value $\langle W \rangle$ of a chemistry-catalog observable.
3. Compute the **cumulant-truncated** estimate $\langle W \rangle_{\mathrm{trunc}}$ that you would get if you only had access to the 1- and 2-RDM.
4. Compute the **bias** $|\langle W \rangle - \langle W \rangle_{\mathrm{trunc}}|$.
5. Compute the **certified bound** $C_W \cdot \Delta$ at all three tightening levels.
6. Verify the theorem: $|\mathrm{bias}| \le C_W \cdot \Delta$ at every level.
7. Slide to the Hartree-Fock baseline and check that $\Delta$ and the bound collapse to exact zero (the worked-example theorem).

**The theorem in one inequality.** For any charge-neutral fermionic word $W$ of length $\le r$ on a $U(1)$-invariant state $\rho$,
$$
\bigl| \langle W \rangle_{\rho} - \langle W \rangle_{\mathrm{trunc}, \rho} \bigr|
\;\le\;
C_W \cdot \Delta_{r, U(1)}^{\mathrm{cat}}(\rho),
$$
where $C_W$ is one of the three integer-valued partition-lattice constants and $\Delta$ is the high-cumulant envelope of $\rho$.

In [ ]:
# Bootstrap on Colab / fresh env (skipped if the library is already importable).
# Local users running after 'uv sync --extra dev' skip the install.
# On Colab the install uses uv pip install --system, which is uv's native
# compatibility subcommand for the system Python environment (not the pip tool).
import importlib.util as _ilu
if _ilu.find_spec("cumulant_residual_cert") is None:
    import subprocess as _sp, os as _os
    # Pin uv to a known directory so we invoke it by absolute path. UV_INSTALL_DIR
    # is the installer's explicit override; without it the location depends on
    # CARGO_HOME / XDG_BIN_HOME inheritance (Colab differs from local Linux).
    _UV_DIR = "/tmp/uv-bootstrap"
    _os.makedirs(_UV_DIR, exist_ok=True)
    _env = {**_os.environ, "UV_INSTALL_DIR": _UV_DIR}
    _sp.check_call(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, executable="/bin/bash", env=_env,
    )
    _UV = f"{_UV_DIR}/uv"
    _sp.check_call([
        _UV, "pip", "install", "--system", "-q",
        "cumulant_residual_cert@git+https://github.com/kootru-repo/cumulant-residual-cert.git",
        "numpy",
    ])

import cumulant_residual_cert
import numpy as np
print(f"cumulant_residual_cert {cumulant_residual_cert.__version__} ready")

## Step 1: Build a small correlated $U(1)$-invariant state

Take a two-electron superposition on $n = 4$ spin-orbitals:
$$
|\psi\rangle = \cos\theta \, |1100\rangle + \sin\theta \, |0011\rangle,
\qquad
\rho = |\psi\rangle \langle \psi |.
$$
Both basis kets have particle number $N = 2$, so $\rho$ commutes with $\widehat N$ ($U(1)$-invariant). For $\theta = \pi/6$ this is correlated (not a single Slater determinant), so the worked-example $\Delta = 0$ result does not apply and we will see a positive bias.

**Expected.** Trace 1; mixture weights $\cos^2(\pi/6) = 0.75$ and $\sin^2(\pi/6) = 0.25$.

In [ ]:
import numpy as np

n = 4
theta = np.pi / 6

psi = np.zeros(2 ** n, dtype=complex)
psi[int('1100', 2)] = np.cos(theta)
psi[int('0011', 2)] = np.sin(theta)
rho = np.outer(psi, psi.conj())

print(f'theta = pi/6 = {theta:.4f} rad')
print(f'Trace check: tr(rho)        = {np.trace(rho).real:.6f}  (expected 1.000000)')
print(f'Weight on |1100>: cos^2 t   = {np.cos(theta)**2:.6f}  (expected 0.750000)')
print(f'Weight on |0011>: sin^2 t   = {np.sin(theta)**2:.6f}  (expected 0.250000)')

## Step 2: Pick a chemistry-catalog observable

The chemistry catalog at $r = 4$ has five words. We pick the worst-case word for tightness, $W = n_1 n_2 n_3 n_4$, because its block-refined constant is the largest in the catalog: $\widehat B^{\mathrm{charge}}_4(W) = 5$.

**Expected.** Word length 4; per-letter charges $(0,0,0,0)$; total charge 0 (catalog admits only charge-neutral words). Three constants on this word: universal $B_4 = 105$, charge-filtered $B^{\mathrm{charge}}_4 = 105$, block-refined $\widehat B^{\mathrm{charge}}_4 = 5$.

In [ ]:
from cumulant_residual_cert import Catalog, constants

catalog = Catalog.chemistry_r4()
word = next(w for w in catalog if w.name == 'n_i n_j n_k n_ell')
sites = (1, 2, 3, 4)

print(f'Word: {word.name}')
print(f'Letters: {word.letters}')
print(f'Per-letter charges: {word.charges}    (sum = {word.total_charge}, charge-neutral)')
print()
print('Three constant levels for this word:')
print(f'  universal        B_4         = {constants.universal(catalog.r)}    (expected 105)')
print(f'  charge-filtered  B^c_4(W)    = {constants.charge_filtered(catalog.r, word)}    (expected 105)')
print(f'  block-refined    Bhat^c_4(W) = {constants.block_refined(catalog.r, word)}    (expected 5)')

## Step 3: True expectation value $\langle W \rangle_{\mathrm{true}}$

Build $W$ as a dense $2^n \times 2^n$ matrix via the library's Jordan-Wigner fermion algebra, then evaluate $\mathrm{tr}(\rho \, W)$.

**Expected.** Both basis kets in $\rho$ have exactly two electrons. The four-site number-product $n_1 n_2 n_3 n_4$ is one only when all four orbitals are occupied. So $\langle n_1 n_2 n_3 n_4 \rangle = 0$ exactly on this two-electron state, regardless of $\theta$.

In [ ]:
from cumulant_residual_cert._fermion import letter_op

def build_word_op(letters, sites, n):
    op = letter_op(letters[0], sites[0], n)
    for L, s in zip(letters[1:], sites[1:]):
        op = op @ letter_op(L, s, n)
    return op

W_dense = build_word_op(word.letters, sites, n)
W_true = complex(np.trace(rho @ W_dense))
print(f'<W>_true = {W_true.real:+.6f}    (expected exactly 0)')

## Step 4: Cumulant-truncated estimate $\langle W \rangle_{\mathrm{trunc}}$

This is the value a chemistry workflow would compute from the 1- and 2-RDM alone (the order-$\le 2$ closure). We build the RDMs from the dense state for ground-truth purposes, then truncate.

The order-$\le 2$ closure sums over partitions $\pi$ of $[4]$ where every block has size $\le 2$; each size-1 block contributes a 1-letter moment ($D^{(1)}$ entry) and each size-2 block contributes a 2-letter connected cumulant.

**Expected.** Number of such partitions: $10$ (1 all-singletons + 6 with one pair + 3 with two pairs = 10, vs. Bell$(4) = 15$ total partitions; the 5 with a block of size $\ge 3$ are dropped). The truncated value will be small but **non-zero**, equal to the 2-RDM-only reconstruction of the 4-number product. The gap $|\langle W \rangle_{\mathrm{true}} - \langle W \rangle_{\mathrm{trunc}}|$ is the truncation bias the theorem certifies.

In [ ]:
from itertools import product as iproduct

from cumulant_residual_cert._rdm_eval import evaluate_subword_moment
from cumulant_residual_cert._partition import set_partitions

def build_rdm(rho, k, n):
    shape = (n,) * (2 * k)
    rdm = np.zeros(shape, dtype=complex)
    for idx in iproduct(range(n), repeat=2 * k):
        p, q = idx[:k], idx[k:]
        op = letter_op('I', 1, n)
        for s0 in p:
            op = op @ letter_op('a_dag', s0 + 1, n)
        for s0 in reversed(q):
            op = op @ letter_op('a', s0 + 1, n)
        rdm[idx] = np.trace(rho @ op)
    return rdm

rdm1 = build_rdm(rho, 1, n)
rdm2 = build_rdm(rho, 2, n)
rdm3 = build_rdm(rho, 3, n)
rdm4 = build_rdm(rho, 4, n)

def order_le_2_closure(word, sites, rdm1, rdm2):
    m = word.length
    partitions_le_2 = [
        pi for pi in set_partitions(list(range(1, m + 1)))
        if all(len(B) <= 2 for B in pi)
    ]
    total = 0.0 + 0j
    for pi in partitions_le_2:
        prod = 1.0 + 0j
        for B in pi:
            B_sorted = tuple(sorted(B))
            sub_letters = tuple(word.letters[i - 1] for i in B_sorted)
            sub_sites = tuple(sites[i - 1] for i in B_sorted)
            if len(B) == 1:
                prod *= evaluate_subword_moment(sub_letters, sub_sites, rdm1)
            else:
                mu12 = evaluate_subword_moment(sub_letters, sub_sites, rdm1, rdm2)
                mu1 = evaluate_subword_moment((sub_letters[0],), (sub_sites[0],), rdm1)
                mu2 = evaluate_subword_moment((sub_letters[1],), (sub_sites[1],), rdm1)
                prod *= mu12 - mu1 * mu2
        total += prod
    return total, len(partitions_le_2)

W_trunc, n_partitions = order_le_2_closure(word, sites, rdm1, rdm2)
print(f'Number of order-<=2 partitions summed: {n_partitions}    (expected 10 of Bell(4) = 15)')
print(f'<W>_trunc = {W_trunc.real:+.6f}    (expected small, nonzero)')
print()
bias = abs(W_true - W_trunc)
print(f'<W>_true  = {W_true.real:+.6f}    (Step 3)')
print(f'<W>_trunc = {W_trunc.real:+.6f}    (this step)')
print(f'bias      = |<W>_true - <W>_trunc| = {bias:+.6f}    <- the quantity the theorem bounds')

## Step 5: The envelope $\Delta$

$$
\Delta_{r, U(1)}^{\mathrm{cat}}(\rho) = \max_{W' \in \mathrm{catalog}} |\kappa_{W'}(\rho)|.
$$
Evaluated directly from RDMs via the Mobius formula. We list every $\kappa_{W'}$ so the max is visible.

**Expected.** A small positive number controlled by the 3- and 4-cumulants of this correlated state. The maximizing word need not be $W$ itself; the envelope is a *catalog-wide* quantity.

In [ ]:
from cumulant_residual_cert._rdm_eval import evaluate_word_cumulant

sites_per_word = {
    'n_i n_j n_k':              (1, 2, 3),
    'a_dag_i a_j n_k':          (1, 2, 3),
    'a_dag_i a_j n_k n_ell':    (1, 2, 3, 4),
    'a_dag_i a_dag_j a_k a_ell': (1, 2, 3, 4),
    'n_i n_j n_k n_ell':        (1, 2, 3, 4),
}
kappa_per_word = {
    w.name: evaluate_word_cumulant(w, sites_per_word[w.name], rdm1, rdm2, rdm3, rdm4)
    for w in catalog
}
for name, kappa in kappa_per_word.items():
    print(f'  |kappa({name:32s})| = {abs(kappa):.6f}')
Delta = max(abs(k) for k in kappa_per_word.values())
print()
print(f'Delta = max |kappa| over the catalog = {Delta:.6f}')

## Step 6: Apply the certified bound

Multiply $\Delta$ by $C_W$ at each of the three tightening levels. The theorem asserts $|\mathrm{bias}| \le C_W \cdot \Delta$ at every level; only the *tightness* changes.

**Expected.** All three bounds are upper bounds on the bias from Step 4. The block-refined bound is the smallest, equal to $5 \Delta$. The universal bound is the largest, equal to $105 \Delta$. The tightening from universal to block-refined is exactly $105/5 = 21\times$ on this worst-case word.

In [ ]:
from cumulant_residual_cert import certify

print(f'Bias observed: |<W>_true - <W>_trunc| = {bias:+.6f}')
print()
for level in ('universal', 'charge_filtered', 'block_refined'):
    result = certify(catalog, delta=Delta, level=level)
    C_W = result.constants_used[word.name]
    bound = result.bounds[word.name]
    ok = bias <= bound + 1e-12
    print(f'  {level:18s}  C_W = {C_W:>3d}  ->  bound = {C_W} * Delta = {bound:+.6f}    '
          f'({"holds" if ok else "VIOLATES"})')
print()
tight_C = constants.block_refined(catalog.r, word)
loose_C = constants.universal(catalog.r)
print(f'Tightening on this word: universal {loose_C} -> block-refined {tight_C} = '
      f'{loose_C // tight_C}x.')

## Step 7: Hartree-Fock baseline collapses everything to exact zero

Replace the correlated state by a single Slater determinant $|1100\rangle$ (an HF baseline in this basis). The worked-example theorem proves $\Delta = 0$ exactly on every occupation-basis diagonal product state. The truncated estimate equals the true value at machine precision, and the certified bound is exactly zero.

**Expected.** $\langle W \rangle_{\mathrm{true}} = 0$, $\langle W \rangle_{\mathrm{trunc}} = 0$, bias = 0, $\Delta = 0$, all bounds = 0.

In [ ]:
psi_HF = np.zeros(2 ** n, dtype=complex); psi_HF[int('1100', 2)] = 1.0
rho_HF = np.outer(psi_HF, psi_HF.conj())
rdm1_HF, rdm2_HF = build_rdm(rho_HF, 1, n), build_rdm(rho_HF, 2, n)
rdm3_HF, rdm4_HF = build_rdm(rho_HF, 3, n), build_rdm(rho_HF, 4, n)

W_true_HF = complex(np.trace(rho_HF @ W_dense))
W_trunc_HF, _ = order_le_2_closure(word, sites, rdm1_HF, rdm2_HF)
Delta_HF = max(
    abs(evaluate_word_cumulant(w, sites_per_word[w.name], rdm1_HF, rdm2_HF, rdm3_HF, rdm4_HF))
    for w in catalog
)
print(f'Hartree-Fock state |1100>:')
print(f'  <W>_true             = {W_true_HF.real:+.2e}    (expected 0)')
print(f'  <W>_trunc            = {W_trunc_HF.real:+.2e}    (expected 0)')
print(f'  bias                 = {abs(W_true_HF - W_trunc_HF):.2e}  (expected 0 at machine precision)')
print(f'  Delta                = {Delta_HF:.2e}            (expected 0 by worked-example theorem)')
print(f'  block-refined bound  = {constants.block_refined(catalog.r, word) * Delta_HF:.2e}  (expected 0)')

## What this means for your workflow

Decision points the certificate supports:

- **Hartree-Fock baseline at zero cost.** Any HF / DFT determinant in its canonical orbital basis is in the Bernoulli class. The certificate returns $\Delta = 0$ exactly and the truncated estimate is the true value. No measurement required.
- **Post-HF chemistry from RDMs.** Your CISD / CASCI / FCI run produced 1-, 2-, optionally 3-, 4-RDM. Pass them to `cumulant_residual_cert.adapters.pyscf.from_rdms(...)` and get the certified bound on every catalog observable without measuring anything additional.
- **Shadow-data certification.** If you only have shadow records (random-Pauli or matchgate / fermionic-Gaussian), use `delta_ucb` or `delta_ucb_from_majorana_moments` to upper-bound $\Delta$ from the data, then run the same `certify(catalog, delta=...)`.
- **Go / no-go on a tolerance.** Pick a target bias tolerance $\epsilon$. If the certified bound is below $\epsilon$, the truncated estimate is safe at $\epsilon$. If not, the closure is **not** certified on this state and the workflow should escalate (longer measurement, higher-order RDM, alternative method).

**Next-step notebooks.**

- `02_bernoulli_worked.ipynb`: random Bernoulli product state at $n = 6$, direct connected-cumulant computation, confirms $\Delta = 0$ at machine precision.
- PySCF mean-field workflow lives in `05_cookbook.ipynb` Recipe 1 (HF / DFT baseline on H$_2$ STO-3G).
- `04_real_shadow_data.ipynb`: pluggable shadow-data routing (simulator + IBM / Rigetti / IonQ / IQM stubs).
- `05_cookbook.ipynb`: nine concrete recipes for applying the certificate to your own state, RDM data, shadow data, custom catalog, or workflow decision.